# 📊 Evaluación de Modelos — FIDE Chess Dataset

**Evaluación 2 — Validación y Evaluación (20%)**

Este notebook **no entrena modelos**. Carga los resultados ya calculados por el
pipeline de Kedro (`evaluation_report`) y genera visualizaciones avanzadas:
1. Gráfico de barras comparando F1-Score y Accuracy por modelo (validación cruzada)
2. Heatmap de la Matriz de Confusión del mejor modelo
3. Tabla resumen de métricas en test

In [ ]:
# ============================================================
# Celda 1: Inicializar sesión de Kedro
# ============================================================
%load_ext kedro.ipython

In [ ]:
# ============================================================
# Celda 2: Imports y configuración de visualización
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo profesional
sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
print('✅ Librerías cargadas correctamente')

In [ ]:
# ============================================================
# Celda 3: Cargar reporte de evaluación desde el catálogo
# ============================================================
# El pipeline 'model_evaluation' ya ejecutó la validación cruzada
# y guardó los resultados en evaluation_report (JSON)
report = catalog.load('evaluation_report')

print('📋 Reporte de evaluación cargado exitosamente')
print(f'   Mejor modelo:    {report["best_model"]}')
print(f'   Modelos comparados: {list(report["model_comparison"].keys())}')
print(f'\n   Métricas en test del mejor modelo:')
for metric, value in report['test_metrics'].items():
    print(f'     {metric:>12s}: {value:.4f}')

---
## 1. Comparación de Modelos — F1-Score y Accuracy (Cross-Validation)

In [ ]:
# ============================================================
# GRÁFICO 1: Barras comparando F1-Score y Accuracy por modelo
# ============================================================
# Extraer métricas de validación cruzada de cada modelo
comparison = report['model_comparison']

model_names = list(comparison.keys())
cv_accuracy = [comparison[m]['cv_accuracy_mean'] for m in model_names]
cv_f1 = [comparison[m]['cv_f1_mean'] for m in model_names]
cv_accuracy_std = [comparison[m]['cv_accuracy_std'] for m in model_names]
cv_f1_std = [comparison[m]['cv_f1_std'] for m in model_names]

# Crear gráfico de barras agrupadas
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(model_names))
width = 0.35

bars_acc = ax.bar(x - width/2, cv_accuracy, width, yerr=cv_accuracy_std,
                  label='Accuracy (CV)', color='#3498db', edgecolor='white',
                  capsize=5, alpha=0.85)
bars_f1 = ax.bar(x + width/2, cv_f1, width, yerr=cv_f1_std,
                 label='F1-Score (CV)', color='#e74c3c', edgecolor='white',
                 capsize=5, alpha=0.85)

# Etiquetas de valor sobre cada barra
for bar in bars_acc:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.008,
            f'{h:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars_f1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.008,
            f'{h:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Comparación de Modelos — Accuracy y F1-Score (Cross-Validation 5-Fold)',
             fontweight='bold', fontsize=15)
ax.set_xlabel('Modelo')
ax.set_ylabel('Score')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=30, ha='right')
ax.set_ylim(0, 1.08)
ax.legend(fontsize=12, loc='lower right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Tabla resumen de validación cruzada
# ============================================================
cv_table = pd.DataFrame({
    'Modelo': model_names,
    'Accuracy (CV)': [f"{comparison[m]['cv_accuracy_mean']:.4f} ± {comparison[m]['cv_accuracy_std']:.4f}"
                      for m in model_names],
    'F1-Score (CV)': [f"{comparison[m]['cv_f1_mean']:.4f} ± {comparison[m]['cv_f1_std']:.4f}"
                      for m in model_names],
}).set_index('Modelo')

print('📋 Resultados de Validación Cruzada (5-Fold):')
display(cv_table)

---
## 2. Matriz de Confusión del Mejor Modelo (Heatmap)

In [ ]:
# ============================================================
# GRÁFICO 2: Heatmap de la Matriz de Confusión
# ============================================================
# Extraer la matriz de confusión del reporte (ya calculada en el pipeline)
cm = np.array(report['confusion_matrix'])
best_model_name = report['best_model']

fig, ax = plt.subplots(figsize=(8, 7))

# Heatmap con anotaciones detalladas
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['No Experto', 'Experto'],
    yticklabels=['No Experto', 'Experto'],
    ax=ax, linewidths=1, linecolor='white',
    annot_kws={'fontsize': 18, 'fontweight': 'bold'},
    cbar_kws={'label': 'Cantidad de Predicciones'}
)

ax.set_title(f'Matriz de Confusión — {best_model_name}',
             fontweight='bold', fontsize=15, pad=15)
ax.set_xlabel('Predicción', fontsize=13)
ax.set_ylabel('Valor Real', fontsize=13)

# Calcular métricas derivadas de la CM
tn, fp, fn, tp = cm.ravel()
total = cm.sum()

# Texto informativo
info_text = (
    f'TP={tp:,}  FP={fp:,}\n'
    f'FN={fn:,}  TN={tn:,}\n'
    f'Total: {total:,}'
)
ax.annotate(
    info_text,
    xy=(1.35, 0.5), xycoords='axes fraction',
    ha='left', va='center', fontsize=11,
    bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.9)
)

plt.tight_layout()
plt.show()

print(f'\n📊 Desglose de la Matriz de Confusión ({best_model_name}):')
print(f'   Verdaderos Positivos  (TP): {tp:>8,}')
print(f'   Verdaderos Negativos  (TN): {tn:>8,}')
print(f'   Falsos Positivos      (FP): {fp:>8,}')
print(f'   Falsos Negativos      (FN): {fn:>8,}')

---
## 3. Métricas del Mejor Modelo en Test

In [ ]:
# ============================================================
# Tabla y gráfico de métricas en test del mejor modelo
# ============================================================
test_metrics = report['test_metrics']

# Tabla estilizada
metrics_df = pd.DataFrame({
    'Métrica': list(test_metrics.keys()),
    'Valor': list(test_metrics.values()),
}).set_index('Métrica')

print(f'📋 Métricas en Test — {best_model_name}:')
display(metrics_df.style.format('{:.4f}').background_gradient(cmap='YlGn'))

# Gráfico radar-like de barras horizontales
fig, ax = plt.subplots(figsize=(10, 5))

metric_names = list(test_metrics.keys())
metric_values = list(test_metrics.values())
colors = sns.color_palette('viridis', n_colors=len(metric_names))

bars = ax.barh(metric_names, metric_values, color=colors,
               edgecolor='white', height=0.6)

for bar, val in zip(bars, metric_values):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', ha='left', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 1.1)
ax.set_title(f'Métricas en Test — {best_model_name}',
             fontweight='bold', fontsize=14)
ax.set_xlabel('Score')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Conclusiones de la Evaluación

- La validación cruzada 5-fold confirma la estabilidad de los modelos.
- La matriz de confusión revela el balance entre falsos positivos y falsos negativos.
- Las métricas en test validan que el modelo generaliza correctamente a datos no vistos.
- El siguiente paso es optimizar los hiperparámetros del mejor modelo.